# 03 — Image Analysis (EDA)

General exploratory data analysis on preprocessed imagery and labels.
**Assumes `02_image_processing` has been run.**

In [ ]:
import sys, os, re
from glob import glob
sys.path.append("../")

# ── S2 source directory (Google Drive mount) ──────────────────────────────────
S2_DIR = "/Users/dikaizm/Library/CloudStorage/GoogleDrive-dikamaah@gmail.com/My Drive/S2_Annual_15d_sacramento_3"

# All S2 files — 2022 (train), 2023 (train), 2024 (test)
ALL_S2 = sorted(glob(os.path.join(S2_DIR, "S2H_20*.tif")))
S2_BY_YEAR = {}
for p in ALL_S2:
    yr = os.path.basename(p).split("_")[1]
    S2_BY_YEAR.setdefault(yr, []).append(p)

print(f"S2 files found across all years:")
for yr, files in sorted(S2_BY_YEAR.items()):
    role = "train" if yr in ("2022","2023") else "test"
    print(f"  {yr} ({role}): {len(files)} files")

# ── Band metadata (11 bands from GEE export) ──────────────────────────────────
BAND_INFO = {
    1:  ("B1",  "Coastal Aerosol",  443),
    2:  ("B2",  "Blue",             490),
    3:  ("B3",  "Green",            560),
    4:  ("B4",  "Red",              665),
    5:  ("B5",  "Red Edge 1",       705),
    6:  ("B6",  "Red Edge 2",       740),
    7:  ("B7",  "Red Edge 3",       783),
    8:  ("B8",  "NIR",              842),
    9:  ("B8A", "Red Edge 4",       865),
    10: ("B11", "SWIR 1",           1610),
    11: ("B12", "SWIR 2",           2190),
}
BAND_RED = 4   # B4 (1-indexed in rasterio)
BAND_NIR = 8   # B8

# ── Processed CDL paths (from 02_image_processing) ────────────────────────────
CDL_FILTERED = {
    "2022": "../data/processed/cdl/cdl_2022_study_area_filtered.tif",
    "2023": "../data/processed/cdl/cdl_2023_study_area_filtered.tif",
    "2024": "../data/processed/cdl/cdl_2024_study_area_filtered.tif",
}

# ── 10 active crop classes (Fallow/Idle 61 → background) ─────────────────────
KEEP_CLASSES = [3, 6, 24, 36, 37, 54, 69, 75, 76, 220]
#   3=Rice, 6=Sunflower, 24=Winter Wheat, 36=Alfalfa, 37=Other Hay,
#   54=Tomatoes, 69=Grapes, 75=Almonds, 76=Walnuts, 220=Plums

# ── Output directory for thesis figures ───────────────────────────────────────
FIGURES_DIR = "../documents/thesis/figures"
os.makedirs(FIGURES_DIR, exist_ok=True)

# ── Representative dates for visualisation (using 2022 as reference year) ─────
REPR_DATES  = ["2022_01_01", "2022_03_17", "2022_05_01",
               "2022_07_30", "2022_09_13", "2022_10_28"]
REPR_LABELS = ["Jan (Dormant)", "Mar (Prep)", "May (Early Growth)",
               "Jul (Peak)", "Sep (Maturation)", "Oct (Harvest)"]
PEAK_DATE      = "2022_07_30"   # Summer peak for band grid
SPECTRAL_DATES = [("2022_01_01","Jan"), ("2022_05_01","May"), ("2022_07_30","Jul")]

def parse_date(path):
    m = re.search(r"S2H_\d{4}_(\d{4})_(\d{2})_(\d{2})\.tif", os.path.basename(path))
    if m:
        return f"{m.group(1)}-{m.group(2)}-{m.group(3)}"
    return None

print(f"\nConfig ready. {len(ALL_S2)} total S2 files.")
print(f"KEEP_CLASSES ({len(KEEP_CLASSES)}): {KEEP_CLASSES}")
print(f"FIGURES_DIR: {os.path.abspath(FIGURES_DIR)}")

---
## Helper Functions

In [ ]:
import numpy as np
import rasterio
from rasterio.enums import Resampling
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from datetime import datetime
from utils.constants import USDA_CDL_NAMES, USDA_CDL_COLORS

plt.rcParams.update({"font.family": "sans-serif", "axes.spines.top": False,
                     "axes.spines.right": False})


def read_ds(path, factor=8):
    """Read raster at 1/factor resolution using average resampling."""
    with rasterio.open(path) as src:
        h = max(1, src.height // factor)
        w = max(1, src.width  // factor)
        data = src.read(
            out_shape=(src.count, h, w),
            resampling=Resampling.average,
        ).astype(np.float32)
    return data


def norm(arr, p_lo=2, p_hi=98):
    """Percentile clip and normalize to [0, 1], NaN → 0."""
    valid = arr[np.isfinite(arr)]
    if len(valid) == 0:
        return np.zeros_like(arr)
    lo, hi = np.percentile(valid, [p_lo, p_hi])
    out = np.clip((arr - lo) / (hi - lo + 1e-9), 0, 1)
    out[~np.isfinite(arr)] = 0
    return out


def make_rgb(data):
    """Build True Color composite: B4 (R), B3 (G), B2 (B), 0-indexed bands."""
    return np.stack([norm(data[3]), norm(data[2]), norm(data[1])], axis=-1)


def parse_date(fpath):
    """Parse acquisition date from filename S2H_2022_YYYY_MM_DD.tif."""
    m = re.search(r"S2H_\d{4}_(\d{4})_(\d{2})_(\d{2})", os.path.basename(fpath))
    return datetime(int(m.group(1)), int(m.group(2)), int(m.group(3))) if m else None


print("Helper functions loaded.")


---
## Figure 1 — True Color RGB (6 Representative Dates)
Visualisasi citra True Color (B4-B3-B2) pada 6 tanggal representatif yang
mencerminkan fase fenologi tanaman sepanjang tahun 2022.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for i, (date_str, label) in enumerate(zip(REPR_DATES, REPR_LABELS)):
    fpath = f"{S2_DIR}/S2H_2022_{date_str}.tif"
    ax = axes[i]

    if not os.path.exists(fpath):
        ax.text(0.5, 0.5, f"Tidak tersedia\n{date_str}",
                ha="center", va="center", transform=ax.transAxes, fontsize=10)
        ax.set_facecolor("#f0f0f0")
        ax.axis("off")
        continue

    data = read_ds(fpath, factor=8)
    ax.imshow(make_rgb(data))
    ax.set_title(label, fontsize=11, fontweight="bold", pad=6)
    ax.axis("off")

fig.suptitle(
    "Citra Sentinel-2 True Color (B4–B3–B2) — Sacramento Valley 2022\n"
    "6 Tanggal Representatif Sepanjang Tahun",
    fontsize=13, fontweight="bold", y=1.02,
)
plt.tight_layout()
savepath = os.path.join(FIGURES_DIR, "s2_rgb_temporal.png")
plt.savefig(savepath, dpi=200, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {savepath}")


---
## Figure 2 — All 11 Bands (Peak Season)
Grid seluruh 11 band spektral pada tanggal puncak musim (Juli 2022),
menunjukkan karakteristik pantulan pada panjang gelombang yang berbeda.

In [ ]:
fpath = f"{S2_DIR}/S2H_2022_{PEAK_DATE}.tif"
data  = read_ds(fpath, factor=6)   # ~933 × 780 px per panel
n_bands = data.shape[0]            # 11

ncols = 4
nrows = (n_bands + ncols - 1) // ncols   # ceil → 3

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 11))
axes = axes.flatten()

for i in range(n_bands):
    band_num = i + 1
    name, desc, wl = BAND_INFO[band_num]

    ax = axes[i]
    ax.imshow(norm(data[i]), cmap="gray", interpolation="nearest")
    ax.set_title(f"{name} – {desc}\n({wl} nm)", fontsize=9, fontweight="bold", pad=4)
    ax.axis("off")

# Hide unused subplot slots
for j in range(n_bands, len(axes)):
    axes[j].axis("off")

date_label = PEAK_DATE.replace("_", "-")
fig.suptitle(
    f"Komposit 11 Band Spektral Sentinel-2 — Sacramento Valley\n{date_label} (Puncak Musim Tanam)",
    fontsize=13, fontweight="bold",
)
plt.tight_layout()
savepath = os.path.join(FIGURES_DIR, "s2_band_grid.png")
plt.savefig(savepath, dpi=200, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {savepath}")


---
## Figure 3 — NDVI Temporal Profile
Profil rata-rata NDVI seluruh citra 2022, menunjukkan pola musiman
pertumbuhan dan panen tanaman di Sacramento Valley.

In [ ]:
ndvi_series = []

for fpath in ALL_S2:
    date = parse_date(fpath)
    if date is None:
        continue
    with rasterio.open(fpath) as src:
        h, w = src.height // 8, src.width // 8
        red = src.read(BAND_RED, out_shape=(1, h, w),
                       resampling=Resampling.average)[0].astype(np.float32)
        nir = src.read(BAND_NIR, out_shape=(1, h, w),
                       resampling=Resampling.average)[0].astype(np.float32)

    valid = np.isfinite(red) & np.isfinite(nir) & ((red + nir) > 0)
    ndvi  = np.where(valid, (nir - red) / (nir + red + 1e-9), np.nan)
    ndvi_series.append((date, float(np.nanmean(ndvi))))
    print(f"  {date.strftime('%Y-%m-%d')}  NDVI={np.nanmean(ndvi):.3f}")

ndvi_series.sort(key=lambda x: x[0])
dates, values = zip(*ndvi_series)

fig, ax = plt.subplots(figsize=(13, 5))

# Growth phase shading
ax.axvspan(datetime(2022, 3, 1),  datetime(2022, 5, 15), alpha=0.08,
           color="limegreen", label="Masa Tanam")
ax.axvspan(datetime(2022, 5, 15), datetime(2022, 9, 1),  alpha=0.08,
           color="gold",      label="Masa Tumbuh")
ax.axvspan(datetime(2022, 9, 1),  datetime(2022, 10, 31), alpha=0.08,
           color="orange",    label="Masa Panen")

ax.plot(dates, values, marker="o", linewidth=2, markersize=6,
        color="#2e7d32", zorder=3)
ax.fill_between(dates, values, alpha=0.12, color="#2e7d32")

ax.set_title("Profil NDVI Temporal — Sacramento Valley 2022",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Tanggal Akuisisi", fontsize=11)
ax.set_ylabel("Rata-rata NDVI", fontsize=11)
ax.set_ylim(0, 0.7)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.35)
plt.tight_layout()

savepath = os.path.join(FIGURES_DIR, "s2_ndvi_temporal.png")
plt.savefig(savepath, dpi=200, bbox_inches="tight")
plt.show()
print(f"\n✅ Saved: {savepath}")


---
## Figure 4 — NDVI Per-Class Time Series
Profil NDVI per kelas tanaman menggunakan label CDL yang telah diproses.
> ⚠️ Membutuhkan `data/processed/cdl/cdl_2022_study_area_filtered.tif` — jalankan `02_image_processing.ipynb` terlebih dahulu.

In [ ]:
if not os.path.exists(CDL_FILTERED["2022"]):
    print("⚠️  CDL filtered not found. Run 02_image_processing.ipynb first.")
else:
    # Read CDL at the same downsampled resolution we'll use for S2
    DS_FACTOR = 8
    with rasterio.open(CDL_FILTERED["2022"]) as src:
        h, w = src.height // DS_FACTOR, src.width // DS_FACTOR
        cdl_ds = src.read(
            1, out_shape=(1, h, w),
            resampling=Resampling.nearest,
        )[0].astype(np.int32)

    ndvi_per_class = {cls: [] for cls in KEEP_CLASSES}

    for fpath in ALL_S2:
        date = parse_date(fpath)
        if date is None:
            continue
        with rasterio.open(fpath) as src:
            red = src.read(BAND_RED, out_shape=(1, h, w),
                           resampling=Resampling.average)[0].astype(np.float32)
            nir = src.read(BAND_NIR, out_shape=(1, h, w),
                           resampling=Resampling.average)[0].astype(np.float32)

        valid = np.isfinite(red) & np.isfinite(nir) & ((red + nir) > 0)
        ndvi  = np.where(valid, (nir - red) / (nir + red + 1e-9), np.nan)

        for cls in KEEP_CLASSES:
            mask = (cdl_ds == cls) & valid
            if mask.any():
                ndvi_per_class[cls].append((date, float(np.nanmean(ndvi[mask]))))

    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))
    cmap = plt.cm.get_cmap("tab20", len(KEEP_CLASSES))

    for i, cls in enumerate(KEEP_CLASSES):
        series = sorted(ndvi_per_class[cls], key=lambda x: x[0])
        if not series:
            continue
        d, v = zip(*series)
        name = USDA_CDL_NAMES.get(cls, str(cls))
        ax.plot(d, v, marker="o", markersize=4, linewidth=1.8,
                label=f"{name} ({cls})", color=cmap(i))

    ax.set_title("Profil NDVI Per Kelas Tanaman — Sacramento Valley 2022",
                 fontsize=13, fontweight="bold")
    ax.set_xlabel("Tanggal Akuisisi", fontsize=11)
    ax.set_ylabel("Rata-rata NDVI", fontsize=11)
    ax.legend(fontsize=8, ncol=2, loc="upper left")
    ax.grid(True, alpha=0.35)
    ax.set_ylim(0, 0.85)
    plt.tight_layout()

    savepath = os.path.join(FIGURES_DIR, "s2_ndvi_per_class.png")
    plt.savefig(savepath, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {savepath}")

---
## Figure 5 — Spectral Profile per Band (3 Seasons)
Rata-rata reflektansi per band untuk tiga tanggal representatif (Januari, Mei, Juli),
menunjukkan perbedaan profil spektral antar musim sebagai justifikasi pendekatan multi-temporal.

In [ ]:
BAND_NAMES = [info[0] for info in BAND_INFO.values()]   # ["B1","B2",...,"B12"]

fig, ax = plt.subplots(figsize=(12, 5))
colors = ["#1565c0", "#f57c00", "#2e7d32"]

for (date_str, label), color in zip(SPECTRAL_DATES, colors):
    fpath = f"{S2_DIR}/S2H_2022_{date_str}.tif"
    if not os.path.exists(fpath):
        print(f"⚠️  Not found: {fpath}")
        continue

    data = read_ds(fpath, factor=4)   # higher res for accuracy
    means = []
    for b in range(data.shape[0]):
        band  = data[b]
        valid = np.isfinite(band) & (band > 0)
        means.append(float(np.mean(band[valid])) if valid.any() else np.nan)

    ax.plot(BAND_NAMES, means, marker="o", linewidth=2, markersize=7,
            label=label, color=color)

ax.set_title("Profil Reflektansi Rata-rata per Band — Sacramento Valley 2022",
             fontsize=13, fontweight="bold")
ax.set_xlabel("Band Spektral", fontsize=11)
ax.set_ylabel("Rata-rata Reflektansi", fontsize=11)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.35)
plt.tight_layout()

savepath = os.path.join(FIGURES_DIR, "s2_spectral_profile.png")
plt.savefig(savepath, dpi=200, bbox_inches="tight")
plt.show()
print(f"✅ Saved: {savepath}")


---
## Figure 6 — Data Coverage per Acquisition Date
Persentase piksel valid (bebas awan / nodata) untuk setiap tanggal akuisisi tahun 2022.
Citra dengan cakupan valid < 70% dikecualikan dari dataset.

In [ ]:
coverage = []   # list of (date, valid_frac %)

for fpath in ALL_S2:
    date = parse_date(fpath)
    if date is None:
        continue
    with rasterio.open(fpath) as src:
        h = max(1, src.height // 8)
        w = max(1, src.width  // 8)
        band = src.read(1, out_shape=(1, h, w),
                        resampling=Resampling.average)[0].astype(np.float32)
    valid_frac = float(np.isfinite(band).mean()) * 100
    coverage.append((date, valid_frac))
    print(f"  {date.strftime('%Y-%m-%d')}  valid={valid_frac:.1f}%")

coverage.sort(key=lambda x: x[0])
dates_c, frac_c = zip(*coverage)
mean_cov = sum(frac_c) / len(frac_c)
print(f"\nMean coverage: {mean_cov:.1f}%")

fig, ax = plt.subplots(figsize=(14, 4))
bar_colors = ["#c62828" if f < 70 else "#2e7d32" for f in frac_c]
ax.bar([d.strftime("%m-%d") for d in dates_c], frac_c,
       color=bar_colors, width=0.7, zorder=2)
ax.axhline(mean_cov, color="#1565c0", linewidth=1.5, linestyle="--",
           label=f"Rata-rata ({mean_cov:.1f}%)", zorder=3)

ax.set_title("Persentase Piksel Valid per Tanggal Akuisisi — Sentinel-2 2022",
             fontsize=12, fontweight="bold")
ax.set_xlabel("Tanggal Akuisisi (MM-DD)", fontsize=11)
ax.set_ylabel("Piksel Valid (%)", fontsize=11)
ax.set_ylim(0, 105)
ax.legend(fontsize=10)
ax.tick_params(axis='x', rotation=45)
ax.grid(True, axis='y', alpha=0.35, zorder=0)
plt.tight_layout()

savepath = os.path.join(FIGURES_DIR, "s2_data_coverage.png")
plt.savefig(savepath, dpi=200, bbox_inches="tight")
plt.show()
print(f"\n✅ Saved: {savepath}")


---
## Figure 7 — CDL Label Map with Legend
Visualisasi raster label CDL 2022 dengan warna resmi USDA untuk 11 kelas target + background.
> ⚠️ Membutuhkan `data/processed/cdl/cdl_2022_study_area_filtered.tif` — jalankan `02_image_processing.ipynb` terlebih dahulu.

In [ ]:
from utils.constants import USDA_CDL_COLORS, USDA_CDL_NAMES

# ── KEEP_CLASSES display order + background ───────────────────────────────────
LEGEND_CLASSES = [
    (0,   "Background"),
    (3,   "Rice"),
    (6,   "Sunflower"),
    (24,  "Winter Wheat"),
    (36,  "Alfalfa"),
    (37,  "Other Hay/Non Alfalfa"),
    (54,  "Tomatoes"),
    (61,  "Fallow/Idle Cropland"),
    (69,  "Grapes"),
    (75,  "Almonds"),
    (76,  "Walnuts"),
    (220, "Plums"),
]
# Override background color to light grey for visibility
_bg_color = "#c8c8c8"

# ── Build per-pixel color LUT (index 0-255 → RGB) ────────────────────────────
lut = np.zeros((256, 3), dtype=np.uint8)
for cid in range(256):
    if cid == 0:
        hex_c = _bg_color
    else:
        hex_c = USDA_CDL_COLORS.get(cid, "#ffffff")
    lut[cid] = [int(hex_c[1:3], 16), int(hex_c[3:5], 16), int(hex_c[5:7], 16)]

if not os.path.exists(CDL_FILTERED["2022"]):
    print("⚠️  CDL filtered not found. Run 02_image_processing.ipynb first.")
else:
    DS_FACTOR = 6
    with rasterio.open(CDL_FILTERED["2022"]) as src:
        h = max(1, src.height // DS_FACTOR)
        w = max(1, src.width  // DS_FACTOR)
        cdl = src.read(1, out_shape=(1, h, w),
                       resampling=Resampling.nearest)[0].astype(np.int32)

    rgb = lut[np.clip(cdl, 0, 255)]   # (H, W, 3)

    # ── Legend patches ────────────────────────────────────────────────────────
    patches = []
    for cid, name in LEGEND_CLASSES:
        hex_c = _bg_color if cid == 0 else USDA_CDL_COLORS.get(cid, "#ffffff")
        patches.append(mpatches.Patch(
            facecolor=hex_c, edgecolor="#555555", linewidth=0.5,
            label=f"{name}  ({cid})"
        ))

    # ── Figure: map + legend below ────────────────────────────────────────────
    fig = plt.figure(figsize=(12, 10))
    ax_map = fig.add_axes([0.0, 0.18, 1.0, 0.82])   # map occupies top 82%
    ax_map.imshow(rgb, interpolation="nearest")
    ax_map.set_title(
        "Data Label CDL 2022 — Sacramento Valley\n(11 Kelas Target + Background)",
        fontsize=13, fontweight="bold", pad=8,
    )
    ax_map.axis("off")

    # Legend placed below the map
    ax_map.legend(
        handles=patches,
        loc="upper center",
        bbox_to_anchor=(0.5, -0.04),
        ncol=4,
        fontsize=9,
        frameon=True,
        framealpha=0.9,
        edgecolor="#aaaaaa",
        title="Legenda Kelas CDL",
        title_fontsize=10,
        handlelength=1.5,
        handleheight=1.2,
    )

    savepath = os.path.join(FIGURES_DIR, "cdl_label_map.png")
    plt.savefig(savepath, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {savepath}")

---
## Figure 8 — CDL Class Area Distribution (Top Classes)
Diagram batang cakupan area per kelas CDL di area studi (hektar), diurutkan dari terbesar.
Menampilkan 30 kelas teratas untuk mendukung tabel distribusi kelas di thesis.

In [ ]:
from utils.constants import USDA_CDL_NAMES, USDA_CDL_COLORS

PIXEL_AREA_HA = 77.8 / 10_000   # 77.8 m²/pixel → hectares

if not os.path.exists(CDL_FILTERED["2022"]):
    # Fall back to full reprojected CDL if filtered not available
    cdl_full = "../data/processed/cdl/cdl_2022_study_area.tif"
    cdl_source = cdl_full if os.path.exists(cdl_full) else None
else:
    cdl_source = CDL_FILTERED["2022"]

if cdl_source is None:
    print("⚠️  CDL not found. Run 02_image_processing.ipynb first.")
else:
    with rasterio.open(cdl_source) as src:
        cdl_full_arr = src.read(1).astype(np.int32)

    unique, counts = np.unique(cdl_full_arr, return_counts=True)

    rows = []
    for cid, cnt in zip(unique, counts):
        if cid == 0:
            continue
        name = USDA_CDL_NAMES.get(int(cid), f"Class {cid}")
        area_ha = cnt * PIXEL_AREA_HA
        rows.append({"class_id": int(cid), "class_name": name,
                     "pixel_count": int(cnt), "area_ha": area_ha})

    import pandas as pd
    df = pd.DataFrame(rows).sort_values("area_ha", ascending=False).reset_index(drop=True)
    TOP_N = 20
    df_top = df.head(TOP_N).copy()

    # Colors from USDA palette
    bar_colors = [USDA_CDL_COLORS.get(int(cid), "#aaaaaa") for cid in df_top["class_id"]]

    fig, ax = plt.subplots(figsize=(11, 8))
    bars = ax.barh(df_top["class_name"][::-1], df_top["area_ha"][::-1],
                   color=bar_colors[::-1], edgecolor="#555555", linewidth=0.4)

    # Annotate bars with area value
    for bar, val in zip(bars, df_top["area_ha"][::-1]):
        ax.text(bar.get_width() + 200, bar.get_y() + bar.get_height() / 2,
                f"{val:,.0f} ha", va="center", fontsize=8)

    ax.set_xlabel("Luas Area (hektar)", fontsize=11)
    ax.set_title(f"Distribusi Kelas CDL di Area Studi — Top {TOP_N} Kelas (2022)",
                 fontsize=12, fontweight="bold")
    ax.grid(True, axis="x", alpha=0.35)
    ax.set_xlim(0, df_top["area_ha"].max() * 1.15)
    plt.tight_layout()

    savepath = os.path.join(FIGURES_DIR, "cdl_class_distribution_area.png")
    plt.savefig(savepath, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"✅ Saved: {savepath}")
    print(df_top[["class_name", "pixel_count", "area_ha"]].to_string(index=False))